# 📊 UEBA Lab — Notebook 04: Dashboard & Validation

**Inputs:** `risk_scores.csv`, `alerts.json`, `flagged_users.csv`, CERT `answers/` folder
**Goal:** Build an interactive SOC dashboard, validate detection performance
against CERT r4.2 ground truth, and quantify the alert fatigue reduction
achieved by LLM filtering.

## 📋 Notebook Sections
| Section | Topic |
|---------|-------|
| 4.0 | Setup & Load All Inputs |
| 4.1 | SOC Dashboard (Plotly + Gradio) |
| 4.2 | Ground Truth Validation |
| 4.3 | Alert Fatigue Comparison |
| 4.4 | Key Takeaways & Production Pathway |

> **⚠️ Before running:** Complete Notebooks 01–03.
> Place the CERT r4.2 `answers/` folder in the same directory as this notebook.

---
## 4.0 Setup & Load All Inputs

In [ ]:
import os
import json
import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import seaborn as sns
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import gradio as gr
from sklearn.metrics import (
    confusion_matrix,
    precision_score,
    recall_score,
    f1_score,
    roc_curve,
    auc,
    average_precision_score,
)
from datetime import datetime

warnings.filterwarnings('ignore')
pd.set_option('display.max_columns', 50)
pd.set_option('display.float_format', '{:.4f}'.format)

# ── Paths ─────────────────────────────────────────────────────
RISK_SCORES_PATH  = 'risk_scores.csv'
ALERTS_JSON_PATH  = 'alerts.json'
FLAGGED_PATH      = 'flagged_users.csv'
ANSWERS_DIR       = './answers'
CERT_DATA_PATH    = './cert_r42'

# ── Alert thresholds (must match Notebook 02) ─────────────────
THRESHOLD_WATCH    = 0.40
THRESHOLD_ELEVATED = 0.65
THRESHOLD_CRITICAL = 0.80

print('✅ Imports and configuration loaded.')

In [ ]:
# ── Load risk_scores.csv ──────────────────────────────────────
risk_scores = pd.read_csv(
    RISK_SCORES_PATH, parse_dates=['date_only']
)
risk_scores = risk_scores.sort_values(
    ['user_id', 'date_only']
).reset_index(drop=True)

print(f'✅ risk_scores.csv  : {risk_scores.shape}')
print(f'   Users            : {risk_scores["user_id"].nunique():,}')
print(f'   Date range       : '
      f'{risk_scores["date_only"].min().date()} → '
      f'{risk_scores["date_only"].max().date()}')

In [ ]:
# ── Load alerts.json ──────────────────────────────────────────
with open(ALERTS_JSON_PATH, 'r', encoding='utf-8') as f:
    alerts_store = json.load(f)

alerts_df = pd.DataFrame(alerts_store['alerts'])
alerts_df['peak_risk_day'] = pd.to_datetime(
    alerts_df['peak_risk_day'], errors='coerce'
)

# Separate actionable vs FP-filtered
fp_mask = (
    (alerts_df['threat_type'] == 'Benign') |
    (alerts_df['false_positive_risk'] == 'High')
)
alerts_actionable = alerts_df[~fp_mask].copy()
alerts_fp         = alerts_df[fp_mask].copy()

print(f'✅ alerts.json      : {len(alerts_df)} total alerts')
print(f'   Actionable       : {len(alerts_actionable)}')
print(f'   FP-filtered      : {len(alerts_fp)}')
print(f'   Parse OK         : {alerts_store["ok_count"]}')
print(f'   Generated at     : {alerts_store["generated_at"]}')

In [ ]:
# ── Load flagged_users.csv ────────────────────────────────────
flagged = pd.read_csv(
    FLAGGED_PATH, parse_dates=['date_only']
)
flagged = flagged.sort_values(
    'risk_score', ascending=False
).reset_index(drop=True)

print(f'✅ flagged_users.csv: {flagged.shape}')

# ── Per-user aggregated risk ──────────────────────────────────
user_risk = (
    risk_scores
    .groupby('user_id')
    .agg(
        max_risk_score  = ('risk_score', 'max'),
        mean_risk_score = ('risk_score', 'mean'),
        critical_days   = ('alert_level',
                           lambda x: (x == 'critical').sum()),
        elevated_days   = ('alert_level',
                           lambda x: (x == 'elevated').sum()),
        watch_days      = ('alert_level',
                           lambda x: (x == 'watch').sum()),
        total_days      = ('risk_score', 'count'),
    )
    .reset_index()
    .sort_values('max_risk_score', ascending=False)
)

# Merge role/department if available
for col in ['role', 'department']:
    if col in risk_scores.columns:
        mapping = (
            risk_scores[['user_id', col]]
            .drop_duplicates('user_id')
        )
        user_risk = user_risk.merge(
            mapping, on='user_id', how='left'
        )

print(f'✅ Per-user risk table: {user_risk.shape}')

---
## 4.1 SOC Dashboard (Plotly + Gradio)

The dashboard has four panels:
1. **Risk Leaderboard** — top users ranked by max daily risk score
2. **Behavioral Timeline Heatmap** — risk score over time per user
3. **LLM Alert Cards** — structured alert details per flagged user
4. **Peer Cohort Radar** — how a user's behavior compares to their role cohort

In [ ]:
# ── Panel 1: Risk Leaderboard ─────────────────────────────────
def plot_leaderboard(top_n: int = 25) -> go.Figure:
    top = user_risk.head(top_n).copy()

    bar_colors = [
        '#FF4444' if s >= THRESHOLD_CRITICAL
        else '#FF6B35' if s >= THRESHOLD_ELEVATED
        else '#FFD700'
        for s in top['max_risk_score']
    ]

    hover_text = [
        f'User: {row.user_id}<br>'
        f'Max Risk: {row.max_risk_score:.4f}<br>'
        f'Critical Days: {int(row.critical_days)}<br>'
        f'Elevated Days: {int(row.elevated_days)}<br>'
        f'Role: {row.get("role", "N/A")}'
        for row in top.itertuples()
    ]

    fig = go.Figure(go.Bar(
        x=top['max_risk_score'],
        y=top['user_id'],
        orientation='h',
        marker_color=bar_colors,
        hovertext=hover_text,
        hoverinfo='text',
        text=top['max_risk_score'].round(3),
        textposition='outside',
        textfont=dict(size=10, color='white'),
    ))

    for thresh, color, label in [
        (THRESHOLD_WATCH,    '#FFD700', 'Watch'),
        (THRESHOLD_ELEVATED, '#FF6B35', 'Elevated'),
        (THRESHOLD_CRITICAL, '#FF4444', 'Critical'),
    ]:
        fig.add_vline(
            x=thresh,
            line_dash='dash',
            line_color=color,
            line_width=1.5,
            annotation_text=label,
            annotation_font_color=color,
            annotation_position='top right',
        )

    fig.update_layout(
        title=f'🔴 User Risk Leaderboard — Top {top_n}',
        xaxis_title='Max Daily Risk Score',
        yaxis_title='User ID',
        height=max(400, top_n * 22),
        plot_bgcolor='#0D1B2A',
        paper_bgcolor='#0D1B2A',
        font_color='white',
        title_font_size=14,
        yaxis={'categoryorder': 'total ascending'},
        xaxis=dict(range=[0, 1.05]),
        margin=dict(l=120, r=60, t=60, b=40),
    )
    return fig


plot_leaderboard(25).show()
print('✅ Risk leaderboard rendered.')

In [ ]:
# ── Panel 2: Behavioral Timeline Heatmap ─────────────────────
def plot_timeline_heatmap(top_n: int = 30) -> go.Figure:
    top_users = user_risk.head(top_n)['user_id'].tolist()

    pivot = (
        risk_scores[risk_scores['user_id'].isin(top_users)]
        .pivot_table(
            index='user_id',
            columns='date_only',
            values='risk_score',
            aggfunc='max',
        )
        .fillna(0)
    )

    # Sort rows by max risk score
    pivot = pivot.loc[
        pivot.max(axis=1).sort_values(ascending=False).index
    ]

    fig = go.Figure(go.Heatmap(
        z=pivot.values,
        x=[str(d)[:10] for d in pivot.columns],
        y=pivot.index.tolist(),
        colorscale='RdYlGn_r',
        zmin=0,
        zmax=1,
        colorbar=dict(
            title='Risk Score',
            titlefont=dict(color='white'),
            tickfont=dict(color='white'),
        ),
        hovertemplate=(
            'User: %{y}<br>'
            'Date: %{x}<br>'
            'Risk: %{z:.4f}<extra></extra>'
        ),
    ))

    fig.update_layout(
        title=f'📅 Behavioral Risk Timeline — Top {top_n} Users',
        xaxis_title='Date',
        yaxis_title='User ID',
        height=max(500, top_n * 20),
        plot_bgcolor='#0D1B2A',
        paper_bgcolor='#0D1B2A',
        font_color='white',
        title_font_size=14,
        xaxis=dict(
            tickangle=-45,
            tickfont=dict(size=8),
            nticks=20,
        ),
    )
    return fig


plot_timeline_heatmap(30).show()
print('✅ Behavioral timeline heatmap rendered.')
print('\n💡 Sustained red bands = persistent anomalous behavior.')
print('   Isolated red cells = single-day spikes (lower priority).')

In [ ]:
# ── Panel 3: LLM Alert Card renderer ─────────────────────────
def render_alert_card(user_id: str) -> str:
    """
    Render a plain-text SOC alert card for a given user.
    Used by both the notebook display and the Gradio interface.
    """
    row = alerts_df[alerts_df['user_id'] == user_id]
    if row.empty:
        return f'⚠️  No alert found for user: {user_id}'

    a = row.iloc[0]

    threat_icons = {
        'Data Exfiltration' : '🔴',
        'Sabotage'          : '🟠',
        'Privilege Abuse'   : '🟡',
        'Reconnaissance'    : '🟣',
        'Benign'            : '🟢',
    }
    icon = threat_icons.get(str(a.get('threat_type', '')), '⚪')

    key_evidence = a.get('key_evidence', [])
    if isinstance(key_evidence, str):
        try:
            key_evidence = json.loads(key_evidence)
        except Exception:
            key_evidence = [key_evidence]

    evidence_lines = '\n'.join(
        f'  • {str(e)[:120]}' for e in key_evidence[:5]
    )

    card = f"""
╔══════════════════════════════════════════════════════════╗
  {icon}  SOC ALERT — {str(a.get('alert_level', '?')).upper()}
╠══════════════════════════════════════════════════════════╣
  User ID       : {a.get('user_id', '?')}
  Role          : {a.get('role', 'N/A')}
  Department    : {a.get('department', 'N/A')}
  Peak Risk Day : {str(a.get('peak_risk_day', '?'))[:10]}
  Risk Score    : {float(a.get('risk_score', 0)):.4f}
  Generated     : {str(a.get('generated_at', '?'))[:19]}
╠══════════════════════════════════════════════════════════╣
  THREAT ASSESSMENT
  Threat Type   : {a.get('threat_type', '?')}
  Confidence    : {a.get('confidence', '?')}
  Kill Chain    : {a.get('kill_chain_stage', '?')}
  FP Risk       : {a.get('false_positive_risk', '?')}
╠══════════════════════════════════════════════════════════╣
  ML SCORES
  Isolation Forest : {float(a.get('iso_score', 0)):.4f}
  Autoencoder      : {float(a.get('ae_score', 0)):.4f}
  LSTM Sequence    : {float(a.get('lstm_score', 0)):.4f}
╠══════════════════════════════════════════════════════════╣
  SUMMARY
  {str(a.get('summary', 'N/A'))}
╠══════════════════════════════════════════════════════════╣
  KEY EVIDENCE
{evidence_lines}
╠══════════════════════════════════════════════════════════╣
  RECOMMENDED ACTION
  {str(a.get('recommended_action', 'N/A'))}
╚══════════════════════════════════════════════════════════╝
"""
    return card.strip()


# Preview top 3 alert cards
for uid in alerts_actionable['user_id'].head(3):
    print(render_alert_card(uid))
    print()

In [ ]:
# ── Panel 4: Peer Cohort Radar Chart ─────────────────────────
def plot_radar(
    user_id: str,
    feature_matrix_path: str = 'feature_matrix.parquet',
) -> go.Figure:
    """
    Radar chart comparing a user's average behavioral features
    against their role cohort average.
    """
    radar_features = [
        'files_accessed', 'after_hours_ratio',
        'usb_event_count', 'external_email_ratio',
        'job_site_visits', 'sensitive_file_ratio',
        'email_count', 'cloud_storage_visits',
    ]

    if not os.path.exists(feature_matrix_path):
        return go.Figure().update_layout(
            title='feature_matrix.parquet not found',
            paper_bgcolor='#0D1B2A',
            font_color='white',
        )

    fm = pd.read_parquet(feature_matrix_path)
    present = [f for f in radar_features if f in fm.columns]
    if not present:
        return go.Figure().update_layout(
            title='No radar features found in feature matrix',
            paper_bgcolor='#0D1B2A',
            font_color='white',
        )

    user_avg = (
        fm[fm['user_id'] == user_id][present]
        .mean()
    )
    if user_avg.empty:
        return go.Figure().update_layout(
            title=f'User {user_id} not found',
            paper_bgcolor='#0D1B2A',
            font_color='white',
        )

    # Get role for cohort
    user_role = None
    if 'role' in fm.columns:
        role_vals = fm[fm['user_id'] == user_id]['role'].dropna()
        if not role_vals.empty:
            user_role = role_vals.iloc[0]

    if user_role and 'role' in fm.columns:
        cohort_avg = (
            fm[
                (fm['role'] == user_role) &
                (fm['user_id'] != user_id)
            ][present].mean()
        )
    else:
        cohort_avg = fm[fm['user_id'] != user_id][present].mean()

    # Normalise both to [0, 1] using cohort max
    combined_max = pd.concat(
        [user_avg, cohort_avg], axis=1
    ).max(axis=1).clip(lower=1e-6)

    user_norm   = (user_avg   / combined_max).fillna(0).tolist()
    cohort_norm = (cohort_avg / combined_max).fillna(0).tolist()

    # Close the radar loop
    categories  = present + [present[0]]
    user_norm  += [user_norm[0]]
    cohort_norm += [cohort_norm[0]]

    fig = go.Figure()
    fig.add_trace(go.Scatterpolar(
        r=user_norm,
        theta=categories,
        fill='toself',
        name=f'{user_id}',
        line_color='#FF4444',
        fillcolor='rgba(255,68,68,0.2)',
    ))
    fig.add_trace(go.Scatterpolar(
        r=cohort_norm,
        theta=categories,
        fill='toself',
        name=f'Role cohort ({user_role or "all users"})',
        line_color='#1E90FF',
        fillcolor='rgba(30,144,255,0.15)',
    ))
    fig.update_layout(
        title=f'👥 Peer Cohort Radar — {user_id}',
        polar=dict(
            bgcolor='#0D1B2A',
            radialaxis=dict(
                visible=True,
                range=[0, 1],
                tickfont=dict(color='white', size=9),
                gridcolor='rgba(255,255,255,0.1)',
            ),
            angularaxis=dict(
                tickfont=dict(color='white', size=9),
                gridcolor='rgba(255,255,255,0.1)',
            ),
        ),
        paper_bgcolor='#0D1B2A',
        font_color='white',
        legend=dict(
            font=dict(color='white'),
            bgcolor='rgba(0,0,0,0)',
        ),
        height=480,
        title_font_size=13,
    )
    return fig


# Preview radar for top-risk user
top_user = user_risk.iloc[0]['user_id']
plot_radar(top_user).show()
print(f'✅ Peer cohort radar rendered for {top_user}.')

In [ ]:
# ── Gradio SOC Dashboard ──────────────────────────────────────
all_flagged_users = sorted(
    alerts_df['user_id'].dropna().unique().tolist()
)

def dashboard_fn(
    selected_user: str,
    leaderboard_n: int,
    timeline_n: int,
):
    """
    Gradio callback: returns all four dashboard panels
    for the selected user and slider values.
    """
    leaderboard_fig = plot_leaderboard(int(leaderboard_n))
    timeline_fig    = plot_timeline_heatmap(int(timeline_n))
    alert_card      = render_alert_card(selected_user)
    radar_fig       = plot_radar(selected_user)
    return leaderboard_fig, timeline_fig, alert_card, radar_fig


with gr.Blocks(
    title='UEBA SOC Dashboard',
    theme=gr.themes.Base(
        primary_hue='red',
        neutral_hue='slate',
    ),
    css='"""
        body { background-color: #0D1B2A; color: white; }
        .gradio-container { background-color: #0D1B2A; }
    """',
) as dashboard:

    gr.Markdown(
        '# 🔐 UEBA SOC Dashboard\n'
        '**CERT r4.2 Insider Threat Detection — '
        'Isolation Forest + Autoencoder + LSTM + LLM**'
    )

    with gr.Row():
        user_dropdown = gr.Dropdown(
            choices=all_flagged_users,
            value=all_flagged_users[0] if all_flagged_users else None,
            label='Select Flagged User',
            scale=2,
        )
        leaderboard_slider = gr.Slider(
            minimum=5, maximum=50, value=25, step=5,
            label='Leaderboard — Top N Users',
            scale=1,
        )
        timeline_slider = gr.Slider(
            minimum=5, maximum=50, value=30, step=5,
            label='Timeline — Top N Users',
            scale=1,
        )
        run_btn = gr.Button(
            '🔍 Analyse', variant='primary', scale=1
        )

    with gr.Row():
        leaderboard_plot = gr.Plot(label='Risk Leaderboard')
        timeline_plot    = gr.Plot(label='Behavioral Timeline')

    with gr.Row():
        alert_card_box = gr.Textbox(
            label='LLM Alert Card',
            lines=28,
            max_lines=35,
            show_copy_button=True,
        )
        radar_plot = gr.Plot(label='Peer Cohort Radar')

    run_btn.click(
        fn=dashboard_fn,
        inputs=[
            user_dropdown,
            leaderboard_slider,
            timeline_slider,
        ],
        outputs=[
            leaderboard_plot,
            timeline_plot,
            alert_card_box,
            radar_plot,
        ],
    )

    # Auto-load on startup
    dashboard.load(
        fn=dashboard_fn,
        inputs=[
            user_dropdown,
            leaderboard_slider,
            timeline_slider,
        ],
        outputs=[
            leaderboard_plot,
            timeline_plot,
            alert_card_box,
            radar_plot,
        ],
    )

print('✅ Gradio dashboard defined.')
print('   Launching on http://localhost:7860 ...')
dashboard.launch(
    server_name='0.0.0.0',
    server_port=7860,
    share=False,
    inbrowser=False,
)

---
## 4.2 Ground Truth Validation

CERT r4.2 ships with an `answers/` folder containing the ground truth
insider threat incidents. We load these, align them with our ML scores,
and compute standard detection metrics.

**Metrics computed:**
- Confusion matrix (TP / FP / TN / FN)
- Precision, Recall, F1 at each alert threshold
- Mean Time To Detect (MTTD) — days from incident start to first alert
- ROC curve and AUC for each model + ensemble

In [ ]:
# ── Load CERT r4.2 ground truth ───────────────────────────────
def load_ground_truth(answers_dir: str) -> pd.DataFrame:
    """
    Load all CSV files from the CERT answers/ directory.
    Returns a DataFrame with columns:
        user_id, date, scenario, insider_flag
    """
    if not os.path.exists(answers_dir):
        raise FileNotFoundError(
            f'❌ answers/ directory not found at: {answers_dir}\n'
            f'   Extract the CERT r4.2 archive and ensure the '
            f'answers/ folder is present.'
        )

    frames = []
    for fname in sorted(os.listdir(answers_dir)):
        if not fname.endswith('.csv'):
            continue
        fpath = os.path.join(answers_dir, fname)
        tmp   = pd.read_csv(fpath)
        tmp.columns = tmp.columns.str.lower().str.strip()
        tmp['source_file'] = fname
        frames.append(tmp)
        print(f'   ✅ {fname:<35} {len(tmp):>6,} rows')

    if not frames:
        raise ValueError(
            f'No CSV files found in {answers_dir}'
        )

    gt_raw = pd.concat(frames, ignore_index=True)

    # Standardise user column
    for cand in ['user', 'user_id', 'employee']:
        if cand in gt_raw.columns:
            gt_raw.rename(
                columns={cand: 'user_id'}, inplace=True
            )
            break

    # Standardise date column
    for cand in ['date', 'start', 'start_date', 'incident_date']:
        if cand in gt_raw.columns:
            gt_raw['date'] = pd.to_datetime(
                gt_raw[cand], errors='coerce'
            )
            break

    gt_raw['insider_flag'] = 1
    return gt_raw[[
        'user_id', 'date', 'insider_flag'
    ] + [
        c for c in ['scenario', 'source_file']
        if c in gt_raw.columns
    ]].drop_duplicates()


print('📂 Loading CERT r4.2 ground truth...\n')
ground_truth = load_ground_truth(ANSWERS_DIR)

print(f'\n✅ Ground truth loaded.')
print(f'   Total insider incidents : {len(ground_truth):,}')
print(f'   Unique insider users    : '
      f'{ground_truth["user_id"].nunique():,}')
if 'scenario' in ground_truth.columns:
    print(f'   Scenarios               : '
          f'{ground_truth["scenario"].nunique()}')
ground_truth.head(5)

In [ ]:
# ── Build user-level ground truth labels ─────────────────────
# A user is a true positive if they appear in the answers/ data
insider_users = set(ground_truth['user_id'].unique())
all_scored_users = set(user_risk['user_id'].unique())

# User-level label vector
user_labels = user_risk[['user_id', 'max_risk_score',
                          'max_iso_score' if 'max_iso_score'
                          in user_risk.columns else 'max_risk_score',
                          ]].copy()

# Recompute per-model max scores at user level from risk_scores
for score_col in ['iso_score', 'ae_score', 'lstm_score', 'risk_score']:
    if score_col in risk_scores.columns:
        agg = (
            risk_scores.groupby('user_id')[score_col]
            .max()
            .reset_index()
            .rename(columns={score_col: f'max_{score_col}'})
        )
        user_labels = user_labels.merge(
            agg, on='user_id', how='left'
        )

user_labels['true_label'] = user_labels['user_id'].apply(
    lambda u: 1 if u in insider_users else 0
)

n_true_insiders = user_labels['true_label'].sum()
n_total_users   = len(user_labels)
n_insiders_caught = len(
    insider_users & all_scored_users
)

print('=' * 55)
print('  GROUND TRUTH ALIGNMENT')
print('=' * 55)
print(f'  Total users scored     : {n_total_users:,}')
print(f'  Known insider users    : {len(insider_users):,}')
print(f'  Insiders in scored set : {n_insiders_caught:,}')
print(f'  Insiders NOT scored    : '
      f'{len(insider_users - all_scored_users):,}')
print(f'  Prevalence             : '
      f'{n_true_insiders/n_total_users*100:.2f}%')
print('=' * 55)

In [ ]:
# ── Confusion matrix at ensemble threshold ────────────────────
y_true = user_labels['true_label'].values
y_pred_score = user_labels['max_risk_score'].fillna(0).values

# Predict positive if max risk score >= THRESHOLD_ELEVATED
y_pred = (y_pred_score >= THRESHOLD_ELEVATED).astype(int)

cm = confusion_matrix(y_true, y_pred)
tn, fp, fn, tp = cm.ravel()

precision = precision_score(y_true, y_pred, zero_division=0)
recall    = recall_score(y_true, y_pred, zero_division=0)
f1        = f1_score(y_true, y_pred, zero_division=0)
fpr_rate  = fp / max(fp + tn, 1)

print('=' * 55)
print(f'  CONFUSION MATRIX '
      f'(threshold = {THRESHOLD_ELEVATED})')
print('=' * 55)
print(f'  True Positives  (TP) : {tp:>6}')
print(f'  False Positives (FP) : {fp:>6}')
print(f'  True Negatives  (TN) : {tn:>6}')
print(f'  False Negatives (FN) : {fn:>6}')
print('─' * 55)
print(f'  Precision            : {precision:.4f}')
print(f'  Recall               : {recall:.4f}')
print(f'  F1 Score             : {f1:.4f}')
print(f'  False Positive Rate  : {fpr_rate:.4f}')
print('=' * 55)

In [ ]:
# ── Confusion matrix heatmap ──────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(13, 5))
fig.patch.set_facecolor('#0D1B2A')

# Confusion matrix
cm_labels = np.array([
    [f'TN\n{tn}', f'FP\n{fp}'],
    [f'FN\n{fn}', f'TP\n{tp}'],
])
cm_values = np.array([[tn, fp], [fn, tp]])

ax = axes[0]
ax.set_facecolor('#0D1B2A')
im = ax.imshow(
    cm_values,
    cmap='Blues',
    aspect='auto',
)
for i in range(2):
    for j in range(2):
        ax.text(
            j, i, cm_labels[i, j],
            ha='center', va='center',
            fontsize=14, fontweight='bold',
            color='white',
        )
ax.set_xticks([0, 1])
ax.set_yticks([0, 1])
ax.set_xticklabels(
    ['Predicted Normal', 'Predicted Insider'],
    color='white', fontsize=10
)
ax.set_yticklabels(
    ['Actual Normal', 'Actual Insider'],
    color='white', fontsize=10
)
ax.set_title(
    f'Confusion Matrix\n'
    f'(threshold={THRESHOLD_ELEVATED})',
    color='white', fontsize=12, fontweight='bold'
)
ax.tick_params(colors='white')
for spine in ax.spines.values():
    spine.set_edgecolor('white')

# Metrics bar chart
ax2 = axes[1]
ax2.set_facecolor('#0D1B2A')
metrics = {
    'Precision': precision,
    'Recall'   : recall,
    'F1 Score' : f1,
    'FPR'      : fpr_rate,
}
colors = ['#1E90FF', '#00C853', '#FFD700', '#FF4444']
bars = ax2.barh(
    list(metrics.keys()),
    list(metrics.values()),
    color=colors,
    edgecolor='none',
)
for bar, val in zip(bars, metrics.values()):
    ax2.text(
        val + 0.01, bar.get_y() + bar.get_height() / 2,
        f'{val:.4f}',
        va='center', color='white', fontsize=11
    )
ax2.set_xlim(0, 1.15)
ax2.set_title(
    'Detection Metrics',
    color='white', fontsize=12, fontweight='bold'
)
ax2.tick_params(colors='white')
ax2.set_facecolor('#0D1B2A')
for spine in ax2.spines.values():
    spine.set_edgecolor('#333')
ax2.xaxis.label.set_color('white')
ax2.yaxis.label.set_color('white')

plt.tight_layout()
plt.savefig(
    'validation_confusion.png',
    dpi=120, bbox_inches='tight',
    facecolor='#0D1B2A'
)
plt.show()
print('✅ Confusion matrix saved → validation_confusion.png')

In [ ]:
# ── Precision / Recall / F1 across thresholds ─────────────────
thresholds = np.arange(0.10, 1.00, 0.05)
threshold_metrics = []

for thresh in thresholds:
    y_p = (y_pred_score >= thresh).astype(int)
    threshold_metrics.append({
        'threshold' : round(thresh, 2),
        'precision' : precision_score(y_true, y_p, zero_division=0),
        'recall'    : recall_score(y_true, y_p, zero_division=0),
        'f1'        : f1_score(y_true, y_p, zero_division=0),
        'alerts'    : int(y_p.sum()),
    })

thresh_df = pd.DataFrame(threshold_metrics)

fig = go.Figure()
for metric, color in [
    ('precision', '#1E90FF'),
    ('recall',    '#00C853'),
    ('f1',        '#FFD700'),
]:
    fig.add_trace(go.Scatter(
        x=thresh_df['threshold'],
        y=thresh_df[metric],
        mode='lines+markers',
        name=metric.capitalize(),
        line=dict(color=color, width=2),
        marker=dict(size=5),
    ))

fig.add_vline(
    x=THRESHOLD_ELEVATED,
    line_dash='dash',
    line_color='#FF6B35',
    annotation_text='Elevated threshold',
    annotation_font_color='#FF6B35',
)
fig.update_layout(
    title='Precision / Recall / F1 vs. Alert Threshold',
    xaxis_title='Risk Score Threshold',
    yaxis_title='Score',
    height=400,
    plot_bgcolor='#0D1B2A',
    paper_bgcolor='#0D1B2A',
    font_color='white',
    title_font_size=13,
    legend=dict(font=dict(color='white')),
    yaxis=dict(range=[0, 1.05]),
)
fig.show()
print('✅ Threshold sweep chart rendered.')

best_f1_row = thresh_df.loc[thresh_df['f1'].idxmax()]
print(f'\n💡 Best F1 threshold: {best_f1_row["threshold"]:.2f}')
print(f'   Precision : {best_f1_row["precision"]:.4f}')
print(f'   Recall    : {best_f1_row["recall"]:.4f}')
print(f'   F1        : {best_f1_row["f1"]:.4f}')
print(f'   Alerts    : {int(best_f1_row["alerts"])}')

In [ ]:
# ── ROC Curves — per model + ensemble ────────────────────────
model_score_cols = {
    'Isolation Forest' : 'max_iso_score',
    'Autoencoder'      : 'max_ae_score',
    'LSTM'             : 'max_lstm_score',
    'Ensemble'         : 'max_risk_score',
}

fig = go.Figure()
roc_colors = {
    'Isolation Forest' : '#1E90FF',
    'Autoencoder'      : '#00C853',
    'LSTM'             : '#DA70D6',
    'Ensemble'         : '#FF4444',
}

roc_results = {}
for model_name, score_col in model_score_cols.items():
    if score_col not in user_labels.columns:
        print(f'   ⚠️  {score_col} not found — skipping {model_name}')
        continue

    scores = user_labels[score_col].fillna(0).values
    fpr, tpr, _ = roc_curve(y_true, scores)
    roc_auc     = auc(fpr, tpr)
    avg_prec    = average_precision_score(y_true, scores)
    roc_results[model_name] = {
        'auc': roc_auc, 'avg_precision': avg_prec
    }

    width = 3 if model_name == 'Ensemble' else 1.8
    dash  = 'solid' if model_name == 'Ensemble' else 'dot'

    fig.add_trace(go.Scatter(
        x=fpr, y=tpr,
        mode='lines',
        name=f'{model_name} (AUC={roc_auc:.3f})',
        line=dict(
            color=roc_colors[model_name],
            width=width,
            dash=dash,
        ),
    ))

# Random baseline
fig.add_trace(go.Scatter(
    x=[0, 1], y=[0, 1],
    mode='lines',
    name='Random (AUC=0.500)',
    line=dict(color='#888888', width=1, dash='dash'),
))

fig.update_layout(
    title='ROC Curves — Per Model + Ensemble',
    xaxis_title='False Positive Rate',
    yaxis_title='True Positive Rate',
    height=480,
    plot_bgcolor='#0D1B2A',
    paper_bgcolor='#0D1B2A',
    font_color='white',
    title_font_size=13,
    legend=dict(font=dict(color='white', size=11)),
    xaxis=dict(range=[0, 1]),
    yaxis=dict(range=[0, 1.02]),
)
fig.show()

print('✅ ROC curves rendered.')
print('\n  AUC Summary:')
for model_name, res in roc_results.items():
    print(f'  {model_name:<20}: '
          f'AUC={res["auc"]:.4f}  '
          f'AvgPrecision={res["avg_precision"]:.4f}')

In [ ]:
# ── Mean Time To Detect (MTTD) ────────────────────────────────
print('⏱️  Computing Mean Time To Detect (MTTD)...\n')

mttd_records = []

for _, gt_row in ground_truth.iterrows():
    uid        = gt_row['user_id']
    incident_date = pd.to_datetime(gt_row['date'], errors='coerce')
    if pd.isna(incident_date):
        continue

    # Find first day this user crossed THRESHOLD_ELEVATED
    user_scores = risk_scores[
        (risk_scores['user_id'] == uid) &
        (risk_scores['risk_score'] >= THRESHOLD_ELEVATED) &
        (risk_scores['date_only'] >= incident_date)
    ].sort_values('date_only')

    if user_scores.empty:
        mttd_records.append({
            'user_id'       : uid,
            'incident_date' : incident_date,
            'first_alert'   : None,
            'mttd_days'     : None,
            'detected'      : False,
        })
    else:
        first_alert = user_scores.iloc[0]['date_only']
        mttd_days   = (first_alert - incident_date).days
        mttd_records.append({
            'user_id'       : uid,
            'incident_date' : incident_date,
            'first_alert'   : first_alert,
            'mttd_days'     : mttd_days,
            'detected'      : True,
        })

mttd_df = pd.DataFrame(mttd_records)

detected     = mttd_df[mttd_df['detected'] == True]
not_detected = mttd_df[mttd_df['detected'] == False]

mttd_mean   = detected['mttd_days'].mean()
mttd_median = detected['mttd_days'].median()
mttd_min    = detected['mttd_days'].min()
mttd_max    = detected['mttd_days'].max()

print('=' * 55)
print('  MEAN TIME TO DETECT (MTTD)')
print('=' * 55)
print(f'  Insider incidents    : {len(mttd_df):,}')
print(f'  Detected             : {len(detected):,} '
      f'({len(detected)/max(len(mttd_df),1)*100:.1f}%)')
print(f'  Not detected         : {len(not_detected):,}')
print(f'  MTTD mean            : {mttd_mean:.1f} days')
print(f'  MTTD median          : {mttd_median:.1f} days')
print(f'  MTTD min             : {mttd_min:.0f} days')
print(f'  MTTD max             : {mttd_max:.0f} days')
print('=' * 55)

In [ ]:
# ── MTTD distribution plot ────────────────────────────────────
if len(detected) > 0:
    fig = make_subplots(
        rows=1, cols=2,
        subplot_titles=[
            'MTTD Distribution (days after incident start)',
            'Detection Rate by Incident'
        ]
    )

    # MTTD histogram
    fig.add_trace(
        go.Histogram(
            x=detected['mttd_days'],
            nbinsx=20,
            marker_color='#1E90FF',
            opacity=0.8,
            name='MTTD (days)',
        ),
        row=1, col=1
    )
    fig.add_vline(
        x=mttd_mean,
        line_dash='dash',
        line_color='#FFD700',
        annotation_text=f'Mean: {mttd_mean:.1f}d',
        annotation_font_color='#FFD700',
        row=1, col=1
    )
    fig.add_vline(
        x=mttd_median,
        line_dash='dot',
        line_color='#00C853',
        annotation_text=f'Median: {mttd_median:.1f}d',
        annotation_font_color='#00C853',
        row=1, col=1
    )

    # Detection rate donut
    fig.add_trace(
        go.Pie(
            labels=['Detected', 'Missed'],
            values=[len(detected), len(not_detected)],
            marker_colors=['#00C853', '#FF4444'],
            hole=0.45,
            name='Detection Rate',
            textinfo='label+percent',
            textfont=dict(color='white', size=12),
        ),
        row=1, col=2
    )

    fig.update_layout(
        height=400,
        plot_bgcolor='#0D1B2A',
        paper_bgcolor='#0D1B2A',
        font_color='white',
        title_text='Mean Time To Detect — CERT r4.2',
        title_font_size=14,
        showlegend=False,
    )
    fig.show()
    print('✅ MTTD distribution rendered.')
else:
    print('⚠️  No detected incidents — MTTD plot skipped.')

In [ ]:
# ── Full validation metrics table ─────────────────────────────
validation_summary = pd.DataFrame([
    {
        'Metric'    : 'True Positives (TP)',
        'Value'     : tp,
        'Note'      : 'Insiders correctly flagged'
    },
    {
        'Metric'    : 'False Positives (FP)',
        'Value'     : fp,
        'Note'      : 'Normal users incorrectly flagged'
    },
    {
        'Metric'    : 'True Negatives (TN)',
        'Value'     : tn,
        'Note'      : 'Normal users correctly cleared'
    },
    {
        'Metric'    : 'False Negatives (FN)',
        'Value'     : fn,
        'Note'      : 'Insiders missed'
    },
    {
        'Metric'    : 'Precision',
        'Value'     : round(precision, 4),
        'Note'      : 'TP / (TP + FP)'
    },
    {
        'Metric'    : 'Recall (Detection Rate)',
        'Value'     : round(recall, 4),
        'Note'      : 'TP / (TP + FN)'
    },
    {
        'Metric'    : 'F1 Score',
        'Value'     : round(f1, 4),
        'Note'      : 'Harmonic mean of Precision & Recall'
    },
    {
        'Metric'    : 'False Positive Rate',
        'Value'     : round(fpr_rate, 4),
        'Note'      : 'FP / (FP + TN)'
    },
    {
        'Metric'    : 'Ensemble AUC-ROC',
        'Value'     : round(roc_results.get(
            'Ensemble', {}).get('auc', 0), 4),
        'Note'      : 'Area under ROC curve'
    },
    {
        'Metric'    : 'MTTD Mean (days)',
        'Value'     : round(mttd_mean, 1) if not np.isnan(mttd_mean) else 'N/A',
        'Note'      : 'Days from incident start to first alert'
    },
    {
        'Metric'    : 'MTTD Median (days)',
        'Value'     : round(mttd_median, 1) if not np.isnan(mttd_median) else 'N/A',
        'Note'      : 'Median days to detection'
    },
])

print('\n📊 Full Validation Metrics Table:')
print(validation_summary.to_string(index=False))

validation_summary.to_csv('validation_metrics.csv', index=False)
print('\n✅ validation_metrics.csv saved.')

---
## 4.3 Alert Fatigue Comparison

Alert fatigue is one of the leading causes of SOC analyst burnout.
We quantify how much the LLM filtering layer reduces the raw alert
volume while preserving detection of true insider threats.

In [ ]:
# ── Alert volume comparison ───────────────────────────────────
# Raw ML alerts = all user-days above each threshold
raw_watch    = (risk_scores['risk_score'] >= THRESHOLD_WATCH).sum()
raw_elevated = (risk_scores['risk_score'] >= THRESHOLD_ELEVATED).sum()
raw_critical = (risk_scores['risk_score'] >= THRESHOLD_CRITICAL).sum()

# Post-LLM alerts = actionable alerts only (FP-filtered removed)
llm_watch    = len(alerts_actionable[
    alerts_actionable['alert_level'].isin(['watch', 'elevated', 'critical'])
])
llm_elevated = len(alerts_actionable[
    alerts_actionable['alert_level'].isin(['elevated', 'critical'])
])
llm_critical = len(alerts_actionable[
    alerts_actionable['alert_level'] == 'critical'
])

# Compute reduction percentages
def pct_reduction(raw, filtered):
    if raw == 0:
        return 0.0
    return (raw - filtered) / raw * 100

print('=' * 65)
print('  ALERT FATIGUE COMPARISON')
print('=' * 65)
print(f'  {"Level":<12} {"Raw ML":>12} {"Post-LLM":>12} {"Reduction":>12}')
print('─' * 65)
for label, raw, filtered in [
    ('Watch+',    raw_watch,    llm_watch),
    ('Elevated+', raw_elevated, llm_elevated),
    ('Critical',  raw_critical, llm_critical),
]:
    red = pct_reduction(raw, filtered)
    print(f'  {label:<12} {raw:>12,} {filtered:>12,} '
          f'{red:>11.1f}%')
print('=' * 65)

# True positive preservation check
insider_set = set(ground_truth['user_id'].unique())
raw_tp_users = set(
    user_risk[
        (user_risk['max_risk_score'] >= THRESHOLD_ELEVATED) &
        (user_risk['user_id'].isin(insider_set))
    ]['user_id']
)
llm_tp_users = set(
    alerts_actionable[
        alerts_actionable['user_id'].isin(insider_set)
    ]['user_id']
)

print(f'\n  True insider users preserved after LLM filtering:')
print(f'  Before LLM : {len(raw_tp_users)} insider users flagged')
print(f'  After LLM  : {len(llm_tp_users)} insider users retained')
if len(raw_tp_users) > 0:
    preservation = len(llm_tp_users) / len(raw_tp_users) * 100
    print(f'  Preservation rate : {preservation:.1f}%')

In [ ]:
# ── Alert fatigue visualisation ───────────────────────────────
fig = make_subplots(
    rows=1, cols=3,
    subplot_titles=[
        'Alert Volume: Raw ML vs Post-LLM',
        'FP Reduction by Alert Level',
        'Insider TP Preservation',
    ]
)

# 1. Grouped bar: raw vs post-LLM
levels = ['Watch+', 'Elevated+', 'Critical']
raw_vals = [raw_watch, raw_elevated, raw_critical]
llm_vals = [llm_watch, llm_elevated, llm_critical]

fig.add_trace(
    go.Bar(
        name='Raw ML',
        x=levels,
        y=raw_vals,
        marker_color='#FF6B35',
        opacity=0.85,
    ),
    row=1, col=1
)
fig.add_trace(
    go.Bar(
        name='Post-LLM',
        x=levels,
        y=llm_vals,
        marker_color='#1E90FF',
        opacity=0.85,
    ),
    row=1, col=1
)

# 2. Reduction % bar
reductions = [
    pct_reduction(raw_watch,    llm_watch),
    pct_reduction(raw_elevated, llm_elevated),
    pct_reduction(raw_critical, llm_critical),
]
fig.add_trace(
    go.Bar(
        x=levels,
        y=reductions,
        marker_color='#00C853',
        text=[f'{r:.1f}%' for r in reductions],
        textposition='outside',
        textfont=dict(color='white'),
        name='Reduction %',
    ),
    row=1, col=2
)

# 3. TP preservation donut
preserved = len(llm_tp_users)
lost      = len(raw_tp_users) - preserved
fig.add_trace(
    go.Pie(
        labels=['Preserved', 'Lost to FP filter'],
        values=[max(preserved, 0), max(lost, 0)],
        marker_colors=['#00C853', '#FF4444'],
        hole=0.45,
        textinfo='label+percent',
        textfont=dict(color='white', size=11),
        name='TP Preservation',
    ),
    row=1, col=3
)

fig.update_layout(
    height=420,
    plot_bgcolor='#0D1B2A',
    paper_bgcolor='#0D1B2A',
    font_color='white',
    title_text='Alert Fatigue Reduction — Raw ML vs LLM-Filtered',
    title_font_size=14,
    barmode='group',
    legend=dict(font=dict(color='white')),
)
fig.show()
print('✅ Alert fatigue comparison rendered.')

In [ ]:
# ── Daily alert volume timeline: raw vs post-LLM ─────────────
daily_raw = (
    risk_scores[risk_scores['risk_score'] >= THRESHOLD_ELEVATED]
    .groupby('date_only')
    .size()
    .reset_index(name='raw_alerts')
)

if 'peak_risk_day' in alerts_actionable.columns:
    daily_llm = (
        alerts_actionable
        .groupby('peak_risk_day')
        .size()
        .reset_index(name='llm_alerts')
        .rename(columns={'peak_risk_day': 'date_only'})
    )
    daily_llm['date_only'] = pd.to_datetime(
        daily_llm['date_only'], errors='coerce'
    )
else:
    daily_llm = pd.DataFrame(
        columns=['date_only', 'llm_alerts']
    )

daily_combined = daily_raw.merge(
    daily_llm, on='date_only', how='left'
).fillna(0)

fig = go.Figure()
fig.add_trace(go.Scatter(
    x=daily_combined['date_only'],
    y=daily_combined['raw_alerts'],
    mode='lines',
    name='Raw ML Alerts',
    line=dict(color='#FF6B35', width=1.5),
    fill='tozeroy',
    fillcolor='rgba(255,107,53,0.15)',
))
fig.add_trace(go.Scatter(
    x=daily_combined['date_only'],
    y=daily_combined['llm_alerts'],
    mode='lines',
    name='Post-LLM Actionable',
    line=dict(color='#1E90FF', width=2),
    fill='tozeroy',
    fillcolor='rgba(30,144,255,0.2)',
))
fig.update_layout(
    title='Daily Alert Volume — Raw ML vs Post-LLM Filtered',
    xaxis_title='Date',
    yaxis_title='Alert Count',
    height=380,
    plot_bgcolor='#0D1B2A',
    paper_bgcolor='#0D1B2A',
    font_color='white',
    title_font_size=13,
    legend=dict(font=dict(color='white')),
)
fig.show()
print('✅ Daily alert volume timeline rendered.')
print('\n💡 The gap between orange and blue = analyst time saved.')

---
## 4.4 Key Takeaways & Production Pathway

This section summarises the lab findings and maps a clear path
from this notebook prototype to a production UEBA deployment on HPE PCAI.

In [ ]:
# ── Print full lab summary ────────────────────────────────────
ensemble_auc = roc_results.get('Ensemble', {}).get('auc', 0)
fp_reduction = pct_reduction(raw_elevated, llm_elevated)

print('=' * 65)
print('  UEBA LAB — COMPLETE RESULTS SUMMARY')
print('  CERT Insider Threat Dataset r4.2')
print('=' * 65)
print()
print('  📦 DATASET')
print(f'  Users scored          : '
      f'{risk_scores["user_id"].nunique():,}')
print(f'  User-days scored      : {len(risk_scores):,}')
print(f'  Known insider users   : {len(insider_users):,}')
print()
print('  🤖 ML MODELS')
for model_name, res in roc_results.items():
    print(f'  {model_name:<22}: '
          f'AUC={res["auc"]:.4f}  '
          f'AvgPrec={res["avg_precision"]:.4f}')
print()
print('  🎯 DETECTION PERFORMANCE (ensemble)')
print(f'  Precision             : {precision:.4f}')
print(f'  Recall                : {recall:.4f}')
print(f'  F1 Score              : {f1:.4f}')
print(f'  AUC-ROC               : {ensemble_auc:.4f}')
print(f'  MTTD Mean             : '
      f'{mttd_mean:.1f} days' if not np.isnan(mttd_mean)
      else '  MTTD Mean             : N/A')
print()
print('  🧠 LLM FILTERING')
print(f'  Alerts before LLM     : {raw_elevated:,}')
print(f'  Alerts after LLM      : {llm_elevated:,}')
print(f'  Alert reduction       : {fp_reduction:.1f}%')
print(f'  TP preservation rate  : '
      f'{len(llm_tp_users)/max(len(raw_tp_users),1)*100:.1f}%')
print()
print('  📁 OUTPUTS')
for f in [
    'feature_matrix.parquet',
    'risk_scores.csv',
    'flagged_users.csv',
    'alerts.json',
    'alerts_summary.csv',
    'validation_metrics.csv',
]:
    exists = '✅' if os.path.exists(f) else '❌'
    print(f'  {exists} {f}')
print('=' * 65)

In [ ]:
# ── Production pathway visualisation ─────────────────────────
stages = [
    {
        'stage'      : '1. Data Ingestion',
        'current'    : 'CERT r4.2 CSV files',
        'production' : 'Kafka / Splunk / Syslog stream',
        'effort'     : 'Medium',
    },
    {
        'stage'      : '2. Feature Engineering',
        'current'    : 'Pandas batch (Notebook 01)',
        'production' : 'Apache Spark / Flink real-time',
        'effort'     : 'High',
    },
    {
        'stage'      : '3. ML Scoring',
        'current'    : 'Sklearn + PyTorch (Notebook 02)',
        'production' : 'MLflow model registry + REST API',
        'effort'     : 'Medium',
    },
    {
        'stage'      : '4. LLM Reasoning',
        'current'    : 'PCAI batch API (Notebook 03)',
        'production' : 'PCAI async inference + caching',
        'effort'     : 'Low',
    },
    {
        'stage'      : '5. SOC Dashboard',
        'current'    : 'Gradio + Plotly (Notebook 04)',
        'production' : 'Grafana / Splunk ES / custom SIEM',
        'effort'     : 'Medium',
    },
    {
        'stage'      : '6. Feedback Loop',
        'current'    : 'Manual validation',
        'production' : 'Analyst labels → model retraining',
        'effort'     : 'High',
    },
]

pathway_df = pd.DataFrame(stages)

effort_colors = {
    'Low'    : '#00C853',
    'Medium' : '#FFD700',
    'High'   : '#FF6B35',
}

print('🗺️  PRODUCTION PATHWAY')
print('=' * 65)
for _, row in pathway_df.iterrows():
    effort_icon = {
        'Low': '🟢', 'Medium': '🟡', 'High': '🟠'
    }.get(row['effort'], '⚪')
    print(f'  {row["stage"]}')
    print(f'    Now        : {row["current"]}')
    print(f'    Production : {row["production"]}')
    print(f'    Effort     : {effort_icon} {row["effort"]}')
    print()
print('=' * 65)

In [ ]:
# ── Key takeaways ─────────────────────────────────────────────
print('''
╔══════════════════════════════════════════════════════════════╗
  🔑  KEY TAKEAWAYS
╠══════════════════════════════════════════════════════════════╣

  1. ENSEMBLE > SINGLE MODEL
     Combining Isolation Forest + Autoencoder + LSTM catches
     threat patterns no single model sees alone. Each model
     contributes complementary signal:
     • ISO  → global point-in-time outliers
     • AE   → subtle reconstruction deviations
     • LSTM → multi-day kill chain sequences

  2. LLM AS ANALYST FORCE-MULTIPLIER
     The PCAI LLM layer reduces alert volume significantly
     while preserving true positive detection. It converts
     raw anomaly scores into analyst-readable evidence
     narratives — reducing investigation time per alert.

  3. SHAP BRIDGES ML AND HUMANS
     Every alert includes the top-5 SHAP feature drivers.
     Analysts know exactly WHY a user was flagged, not just
     that they were flagged.

  4. ROLE-ADJUSTED BASELINES ARE ESSENTIAL
     After-hours activity is normal for some roles (IT, ops).
     Without cohort-adjusted z-scores, these users generate
     chronic false positives.

  5. MTTD IS THE OPERATIONAL METRIC THAT MATTERS
     AUC-ROC measures ranking quality. MTTD measures how
     quickly the system catches real threats — the metric
     that determines actual business impact.

  6. FEEDBACK LOOP CLOSES THE LOOP
     Analyst verdicts (TP / FP) should feed back into model
     retraining. Without this, model performance drifts as
     user behavior evolves.

╚══════════════════════════════════════════════════════════════╝
''')

print('🎯 All four notebooks complete.')
print('   The full UEBA pipeline is operational.')

---
## ✅ Notebook 04 — Summary

### What We Built
| Step | Output |
|------|--------|
| Risk leaderboard | Top-N users ranked by max daily risk score |
| Behavioral timeline heatmap | Risk score over time per user |
| LLM alert card renderer | Structured SOC alert display per user |
| Peer cohort radar chart | User vs. role cohort behavioral comparison |
| Gradio SOC dashboard | Interactive 4-panel analyst interface |
| Ground truth validation | Confusion matrix, Precision/Recall/F1, AUC-ROC |
| MTTD analysis | Mean days from incident start to first alert |
| Alert fatigue comparison | Raw ML vs. LLM-filtered alert volume |
| `validation_metrics.csv` | Full metrics table |

### Complete Pipeline Summary
| Notebook | Role | Key Output |
|----------|------|------------|
| 01 — Data Ingestion | Feature engineering | `feature_matrix.parquet` |
| 02 — ML Detection | Anomaly scoring | `risk_scores.csv`, `flagged_users.csv` |
| 03 — LLM Reasoning | Alert generation | `alerts.json`, `alerts_summary.csv` |
| 04 — Dashboard & Validation | SOC interface + metrics | `validation_metrics.csv` |

### Production Next Steps
1. **Stream ingestion** — replace CSV batch with Kafka/Splunk connector
2. **Model registry** — deploy models via MLflow with versioning
3. **PCAI async inference** — move LLM calls to async queue for scale
4. **SIEM integration** — push alerts to Splunk ES / Microsoft Sentinel
5. **Analyst feedback loop** — capture TP/FP verdicts for retraining
6. **Role-aware thresholds** — per-role alert thresholds based on cohort baselines